In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from fuzzywuzzy import process

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    tickers = []
    for row in table.find_all('tr')[1:]:
        ticker = row.find_all('td')[0].text.strip().replace('.', '-')
        tickers.append(ticker)
    return sorted(tickers)

def map_tickers_to_figis(tickers, api_key):
    headers = {
        'Content-Type': 'application/json',
        'X-OPENFIGI-APIKEY': api_key
    }
    url = 'https://api.openfigi.com/v3/mapping'
    results = []

    for i in range(0, len(tickers), 100):
        batch = tickers[i:i+100]
        mapping_jobs = [{'idType': 'TICKER', 'idValue': ticker, 'exchCode': 'US'} for ticker in batch]
        response = requests.post(url, headers=headers, json=mapping_jobs)
        if response.status_code == 200:
            data = response.json()
            for i, item in enumerate(data):
                if 'data' in item and item['data']:
                    for entry in item['data']:
                        results.append({
                            'ticker': batch[i],
                            'name': entry.get('name'),
                            'figi': entry.get('figi'),
                            'cusip': entry.get('cusip')
                        })
        else:
            print(f"Error: {response.status_code} - {response.text}")
        time.sleep(1)

    override = [
        {'ticker': 'BRK-B', 'cusip': None, 'name': 'BERKSHIRE HATHAWAY INC', 'figi': 'BBG000DWG505'},
        {'ticker': 'BF-B', 'cusip': None, 'name': 'BROWN FORMAN CORP', 'figi': 'BBG001S5P4H1'}
    ]

    for item in override:
        results.append({
            'ticker': item['ticker'],
            'cusip': item['cusip'],
            'name': item['name'],
            'figi': item['figi']
        })

    return pd.DataFrame(results)

def match_names_to_issuer(openfigi_df, csv_df, df_cusip):
    matched = []
    csv_names = csv_df['NAMEOFISSUER'].dropna().tolist()

    for _, row in openfigi_df.iterrows():
        figi_name = row['figi']
        ticker = row['ticker']
        n = row['name']
        if not n:
            best_match = None
        else:
            best_match, score = process.extractOne(n, csv_names)
            if score < 80:  # You can adjust the threshold
                best_match = None
        matched.append(best_match)
        r = csv_df[csv_df['NAMEOFISSUER'] == best_match]
        cusip = r.iloc[-1]['CUSIP']
        figi = r.iloc[-1]['FIGI']
        #name = r.iloc[-1]['NAMEOFISSUER']
        df_cusip.loc[len(df_cusip)] = [cusip, ticker, figi_name,
                                       n]
        print(f"Processed figi {figi_name} - {figi}, cusip {cusip}, ticker {ticker}, name {n} - {best_match}, row {_}.")

    openfigi_df['matched_issuer'] = matched
    return openfigi_df

df_cusip = pd.DataFrame(columns=['CUSIP', 'ticker', 'FIGI', 'name'])

# Example usage
api_key = '8c4a6a49-dc82-4baf-bdb2-27e32ab5c717'
sp500_tickers = get_sp500_tickers()
print(f"Fetched {len(sp500_tickers)} S&P 500 tickers")
figi_df = map_tickers_to_figis(sp500_tickers, api_key)
csv_df = pd.read_csv('sp500_random.csv', dtype={'FIGI': str})

result_df = match_names_to_issuer(figi_df, csv_df, df_cusip)

# Check which tickers were not matched
unmatched = result_df[result_df['matched_issuer'].isna()]
print(f"Unmatched tickers: {len(unmatched)}")
print(unmatched[['ticker', 'name']])

# Optionally: Save to CSV
df_cusip.to_csv('matching.csv', index=False)

Fetched 503 S&P 500 tickers
Processed figi BBG000C2V3D6 - nan, cusip 00846U101, ticker A, name AGILENT TECHNOLOGIES INC - AGILENT TECHNOLOGIES INC, row 0.
Processed figi BBG000B9XRY4 - nan, cusip 037833100, ticker AAPL, name APPLE INC - Apple Inc., row 1.
Processed figi BBG0025Y4RY4 - nan, cusip 00287Y109, ticker ABBV, name ABBVIE INC - Abbvie Inc, row 2.
Processed figi BBG001Y2XS07 - nan, cusip 009066101, ticker ABNB, name AIRBNB INC-CLASS A - AIRBNB INC, row 3.
Processed figi BBG000B9ZXB4 - nan, cusip 002824100, ticker ABT, name ABBOTT LABORATORIES - Abbott Laboratories, row 4.
Processed figi BBG000HXNN20 - nan, cusip G0450A105, ticker ACGL, name ARCH CAPITAL GROUP LTD - Arch Capital Group Ltd., row 5.
Processed figi BBG000D9D830 - nan, cusip G1151C101, ticker ACN, name ACCENTURE PLC-CL A - ACCENTURE PLC CL A, row 6.
Processed figi BBG000BB5006 - nan, cusip 00724F101, ticker ADBE, name ADOBE INC - Adobe Inc, row 7.
Processed figi BBG000BB6G37 - nan, cusip 032654105, ticker ADI, name 

In [96]:
import pandas as pd
from fuzzywuzzy import process

def match_names_to_issuer(openfigi_df, csv_df):
    matched = []
    csv_names = csv_df['NAMEOFISSUER'].dropna().tolist()

    for _, row in openfigi_df.iterrows():
        figi_name = row['figi']
        ticker = row['ticker']
        n = row['name']
        if not n:
            best_match = None
        else:
            best_match, score = process.extractOne(n, csv_names)
            if score < 80:  # You can adjust the threshold
                best_match = None
        matched.append(best_match)
        r = csv_df[csv_df['NAMEOFISSUER'] == best_match]
        print(r)
        cusip = r.iloc[-1]['CUSIP']
        figi = r.iloc[-1]['FIGI']
        #name = r.iloc[-1]['NAMEOFISSUER']
        #df_cusip.loc[len(df_cusip)] = [cusip, ticker, figi_name,
        #                              n]
        print(f"Processed figi {figi_name} - {figi}, cusip {cusip}, ticker {ticker}, name {n} - {best_match}, row {_}.")

    openfigi_df['matched_issuer'] = matched
    return openfigi_df

figi_df = pd.DataFrame([{'ticker': 'NWS',
            'cusip': '65249B109',
            'name': 'VERISIGN INC',
           'figi': 'BBG0035M2ZB7'}])
csv_df = pd.read_csv('sp500_random.csv', dtype={'FIGI': str})
csv_df[csv_df['CUSIP'] == '92343E102']
#csv_df[csv_df['NAMEOFISSUER'] == 'NEWS CORP - CLASS B']
#result_df = match_names_to_issuer(figi_df, csv_df)

,ACCESSION_NUMBER,FILING_DATE,SUBMISSIONTYPE,CIK,PERIODOFREPORT,NAMEOFISSUER,TITLEOFCLASS,CUSIP,FIGI,VALUE,SSHPRNAMT,SSHPRNAMTTYPE,PUTCALL,source_file
310,0000936944-13-000006,2013-08-05,13F-HR,936944,2013-06-30,Verisign Inc,COM,92343E102,NaN,6170,138134,SH,NaN,2013q3_form13f.csv
2125,0001532155-13-000114,2013-08-13,13F-HR,1109448,2013-06-30,VERISIGN INC,COM,92343E102,NaN,13269,297105,SH,NaN,2013q3_form13f.csv
2994,0001398344-13-003924,2013-08-15,13F-HR/A,1510387,2013-06-30,VERISIGN INC,COM,92343E102,NaN,3524,78901,SH,NaN,2013q3_form13f.csv
3054,0001527488-13-000004,2013-08-12,13F-HR,1527488,2013-06-30,VERISIGN INC,COM,92343E102,NaN,22225,497638,SH,NaN,2013q3_form13f.csv
3055,0001527488-13-000004,2013-08-12,13F-HR,1527488,2013-06-30,VERISIGN INC,COM,92343E102,NaN,2238,50118,SH,NaN,2013q3_form13f.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219671,0001398344-25-002899,2025-02-14,13F-HR,1510387,2024-12-31,VERISIGN INC,COM,92343E102,NaN,11813691,57082,SH,NaN,2025q1_form13f.csv
220105,0001527488-25-000002,2025-01-28,13F-HR,1527488,2024-12-31,VERISIGN INC,COM,92343E102,NaN,41,200,SH,NaN,2025q1_form13f.csv
221418,0001172661-25-000413,2025-01-29,13F-HR,1704404,2024-12-31,VERISIGN INC,COM,92343E102,NaN,203649,984,SH,NaN,2025q1_form13f.csv
221826,0001730033-25-000001,2025-01-27,13F-HR,1730033,2024-12-31,VERISIGN INC,COM,92343E102,NaN,26491,128,SH,NaN,2025q1_form13f.csv
